Fonction conditionnelle de concaténation des informations temporelles contenues en ligne par comparaison avec la ligne précédente au sein d’un group by

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [ ]:
#chargement de la table exemple
df=pd.read_csv("path/table_exemple.csv",sep=';')

In [3]:
#remplissage par propagation forward des valeurs vides et traitement des dates et création des périodes temporelles
columns_sort=["id","dateeffet"]
columns_features=["variable1","variable2","variable3","variable4","variable5"]

df = (df.sort_values(columns_sort,ascending=[True, False]).reset_index(drop=True))
		
df[columns_features] = df[columns_features].bfill()

df=df.groupby(["id","dateeffet"]).first().reset_index()

df["datedebut"]=pd.to_datetime(df["dateeffet"],errors='coerce')
df["datefin"] = pd.to_datetime(
                                np.where(df["id"] != df["id"].shift(-1),
                                         pd.to_datetime("today").date(),
                                         df['datedebut'].shift(-1) - timedelta(days=1)
                                         )
)                     



In [4]:
#au sein du même id concaténation des informations temporelles contenues en ligne par comparaison avec la ligne précédente 
df_copy=df
df_copy = (df_copy.drop(columns=["dateeffet"])
        .assign(group=lambda x: (
            (x["id"] != x["id"].shift(1)) |
            (x["variable1"] != x["variable1"].shift(1)) |
            (x["variable2"] != x["variable2"].shift(1)) |
            (x["variable3"] != x["variable3"].shift(1)) |
            (x["variable4"] != x["variable4"].shift(1)) |
            (x["variable5"] != x["variable5"].shift(1)) |
            ((x["datedebut"] - x["datefin"].shift(1)).dt.days != 1)
        ).cumsum())
        .groupby('group', as_index=False)
        .agg({
            "id": "first",
            "variable1":  "first",
            "variable2":  "first",
            "variable3":  "first",
            "variable4":  "first",
            "variable5":  "first",
            "datedebut":  "min",
            "datefin":    "max"
        })
        .drop(columns=["group"])
)
        

In [5]:
df_copy

,id,variable1,variable2,variable3,variable4,variable5,datedebut,datefin
0,id_001,v13,v21,v31,v42,v51,1980-01-01,1995-12-20
1,id_001,v11,v22,v31,v42,v51,1995-12-21,2014-06-06
2,id_001,v11,v23,v32,v42,v51,2014-06-07,2015-08-10
3,id_001,v12,v24,v32,v41,v51,2015-08-11,2026-04-20
4,id_002,v13,v25,v33,v42,v52,2016-01-01,2026-04-20
5,id_003,v14,None,None,None,None,2000-01-01,2012-10-20
6,id_003,v14,None,None,None,v53,2012-10-21,2026-04-20
